In [13]:
import os
import json
from datasets import load_dataset

In [2]:
taco_dataset = load_dataset('/home/kaixin/Desktop/mmcode/TACO')

/home/kaixin/anaconda3/envs/mmcode/lib/python3.11/site-packages/datasets/load.py:922: FutureWarning: The repository for TACO contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at /home/kaixin/Desktop/mmcode/TACO/TACO.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


In [3]:
print(taco_dataset["train"][0].keys())

dict_keys(['question', 'solutions', 'starter_code', 'input_output', 'difficulty', 'raw_tags', 'name', 'source', 'tags', 'skill_types', 'url', 'Expected Auxiliary Space', 'time_limit', 'date', 'picture_num', 'memory_limit', 'Expected Time Complexity'])


In [4]:
taco_dataset["train"][18996]

{'question': "Wherever the destination is, whoever we meet, let's render this song together.\n\nOn a Cartesian coordinate plane lies a rectangular stage of size w × h, represented by a rectangle with corners (0, 0), (w, 0), (w, h) and (0, h). It can be seen that no collisions will happen before one enters the stage.\n\nOn the sides of the stage stand n dancers. The i-th of them falls into one of the following groups:   Vertical: stands at (x_{i}, 0), moves in positive y direction (upwards);  Horizontal: stands at (0, y_{i}), moves in positive x direction (rightwards).  [Image] \n\nAccording to choreography, the i-th dancer should stand still for the first t_{i} milliseconds, and then start moving in the specified direction at 1 unit per millisecond, until another border is reached. It is guaranteed that no two dancers have the same group, position and waiting time at the same time.\n\nWhen two dancers collide (i.e. are on the same point at some time when both of them are moving), they 

In [5]:
taco_dataset_dict = {}

for index, item in enumerate(taco_dataset["train"]):
    identifier = f"train_{index}"
    spec = item["question"]
    taco_dataset_dict[identifier] = spec

for index, item in enumerate(taco_dataset["test"]):
    identifier = f"test_{index}"
    spec = item["question"]
    taco_dataset_dict[identifier] = spec


In [6]:
len(taco_dataset_dict)

26443

In [7]:
def find_problems_with_prefix(folder_path, prefix):
    # List to store folders with 'images' subfolder
    folders = []

    # Iterate over each folder path
    for subfolder in os.listdir(folder_path):
        if not subfolder.startswith(prefix):
            continue
        subfolder_path = os.path.join(folder_path, subfolder)
        folders.append(subfolder_path)

    return folders

az_problems = find_problems_with_prefix("/home/kaixin/Desktop/mmcode/mmcode_dataset", "az")
cf_problems = find_problems_with_prefix("/home/kaixin/Desktop/mmcode/mmcode_dataset", "cf")

In [9]:
len(az_problems), len(cf_problems)

(694, 1941)

In [10]:
def load_crawled_problems(folder_path):
    """
    Function to load data from a JSON file.

    Parameters:
    file_path (str): The path to the 'data.json' file.

    Returns:
    dict: The data loaded from the JSON file.
    """
    with open(os.path.join(folder_path, "data.json"), 'r') as file:
        data = json.load(file)
        return data

In [12]:
load_crawled_problems("/home/kaixin/Desktop/mmcode/mmcode_dataset/cf_500_E")

{'url': 'https://codeforces.com/problemset/problem/500/E',
 'contest_id': 500,
 'problem_index': 'E',
 'time_limit': '2 seconds',
 'memory_limit': '256 megabytes',
 'input_spec': 'The first line contains an integer n (2\u2009≤\u2009n\u2009≤\u20092\u2009×\u2009105)— the number of dominoes.\nNext n lines describe the dominoes. The i-th line (1\u2009≤\u2009i\u2009≤\u2009n) contains two space-separated integers pi, li (1\u2009≤\u2009pi,\u2009li\u2009≤\u2009109)— the x-coordinate and the length of the i-th domino. It is guaranteed that p1\u2009<\u2009p2\u2009<\u2009...\u2009<\u2009pn\u2009-\u20091\u2009<\u2009pn.\nThe next line contains an integer q (1\u2009≤\u2009q\u2009≤\u20092\u2009×\u2009105) — the number of plans.\nNext q lines describe the plans. The j-th line (1\u2009≤\u2009j\u2009≤\u2009q) contains two space-separated integers xj, yj (1\u2009≤\u2009xj\u2009<\u2009yj\u2009≤\u2009n). It means the j-th plan is, to push the xj-th domino, and shoot a video until the yj-th domino falls.',

In [11]:
az_dataset_dict = {path: load_crawled_problems(path)["question"] for path in az_problems}
cf_dataset_dict = {path: load_crawled_problems(path)["question"] for path in cf_problems}

## Matching

In [31]:
import re
# Build id to url map
id_to_problem = {}

def iterate_and_build_id_to_problem_map(dataset_dict):
    for problem in dataset_dict:
        if "url" not in problem or problem["url"] is None:
            continue
        if "codeforces" in problem["url"]:
            match = re.search(r"problem/(\d+)/([A-Za-z]+)", problem["url"])
            if match:
                problem_id = match.group(1)
                problem_index = match.group(2)
                id_to_problem[f"cf_{problem_id}_{problem_index}"] = problem
            else:
                print("Wrong:", problem["url"])
        elif "aizu" in problem["url"]:
            print(problem["url"])
        else:
            continue
        
        



iterate_and_build_id_to_problem_map(taco_dataset["train"])
iterate_and_build_id_to_problem_map(taco_dataset["test"])

# def find_problem_in_taco(problem_id, problem_index):
#     for problem in taco_dataset["train"]:
#         if not problem.get("url"):
#             continue
#         if f"{problem_id}/{problem_index}" in problem["url"]:
#             return problem
        
#     for problem in taco_dataset["train"]:
#         if not problem.get("url"):
#             continue
#         if f"{problem_id}/{problem_index}" in problem["url"]:
#             return problem
#     return None

Wrong: https://codeforces.com/problemset/problem/921/01
Wrong: https://codeforces.com/problemset/problem/0/


In [36]:
pairs = {k:v["url"] for k, v in id_to_problem.items()}
print(len(pairs))
print(pairs)

8561
{'cf_1063_C': 'https://codeforces.com/problemset/problem/1063/C', 'cf_1057_C': 'https://codeforces.com/problemset/problem/1057/C', 'cf_13_E': 'https://codeforces.com/problemset/problem/13/E', 'cf_1492_B': 'https://codeforces.com/problemset/problem/1492/B', 'cf_864_F': 'https://codeforces.com/problemset/problem/864/F', 'cf_957_D': 'https://codeforces.com/problemset/problem/957/D', 'cf_77_B': 'https://codeforces.com/problemset/problem/77/B', 'cf_1312_E': 'https://codeforces.com/problemset/problem/1312/E', 'cf_601_D': 'https://codeforces.com/problemset/problem/601/D', 'cf_132_B': 'https://codeforces.com/problemset/problem/132/B', 'cf_593_C': 'https://codeforces.com/problemset/problem/593/C', 'cf_85_E': 'https://codeforces.com/problemset/problem/85/E', 'cf_57_D': 'https://codeforces.com/problemset/problem/57/D', 'cf_1019_E': 'https://codeforces.com/problemset/problem/1019/E', 'cf_1211_H': 'https://codeforces.com/problemset/problem/1211/H', 'cf_114_B': 'https://codeforces.com/problemse

In [38]:
len(cf_problems)

1941

In [39]:
def read_crawled_info(problem_path):
    solution_path = os.path.join(problem_path, "submissions", "python", "OK.json")
    if os.path.exists(solution_path):
        with open(solution_path, 'r') as f:
            solutions = json.loads(f.read())
            solutions = json.dumps([sol["code"] for sol in solutions])
    else:
        solutions = json.dumps([])

    io_path = os.path.join(problem_path, "input_output.json")
    with open(io_path, 'r') as f:
        _ios = json.loads(f.read())
    ios = {"inputs": [], "outputs": []}
    for s in _ios:
        ios["inputs"].append(s["input"])
        ios["outputs"].append(s["output"])
    ios = json.dumps(ios)
    return {"solutions": solutions, "input_output": ios}

In [48]:
crawled_data_root = "/home/kaixin/Desktop/mmcode/crawl/crawled/codeforces/problems"
for path in cf_problems:
    data = load_crawled_problems(path)
    problem_id, problem_index = path.split("_")[-2:]
    problem_identifier = f"cf_{problem_id}_{problem_index}"

    taco_fields = {
        "solutions": None,
        "input_output": None,
        "difficulty": None,
        "tags": None,
        "skill_types": None,
        "raw_tags": None,
        "Expected Auxiliary Space": None,
        "Expected Time Complexity": None
    }
    # Use the newly crawled data as default value
    crawled_problem_path = os.path.join(crawled_data_root, problem_identifier)
    crawled_taco_fields = read_crawled_info(crawled_problem_path)
    taco_fields.update(crawled_taco_fields)
    
    # If the problem is found in TACO, use their data
    if problem_identifier in id_to_problem:    
        taco_data = id_to_problem[problem_identifier]
        # Copy data from taco
        if taco_data["solutions"]:
            taco_fields["solutions"] = taco_data["solutions"]
        if taco_data["input_output"]:
            taco_fields["input_output"] = taco_data["input_output"]
        taco_fields["difficulty"] = taco_data["difficulty"]
        taco_fields["tags"] = taco_data["tags"]
        taco_fields["skill_types"] = taco_data["skill_types"]
        taco_fields["raw_tags"] = taco_data["raw_tags"]
        taco_fields["Expected Auxiliary Space"] = taco_data["Expected Auxiliary Space"]
        taco_fields["Expected Time Complexity"] = taco_data["Expected Time Complexity"]
    # else:
        # print("Problem not found in TACO", problem_identifier)
        # print(len(json.loads(taco_fields["input_output"])["inputs"]))
        # if not json.loads(taco_fields["input_output"])["inputs"]:
        #     print("No input output", problem_identifier)
        #     print(taco_fields)
    data.update(taco_fields)
    with open(os.path.join(path, "data.json"), 'w') as f:
        f.write(json.dumps(data, indent=4))

        




In [17]:
for path, taco_data in results:
    filename = os.path.join(path, "data.json")
    with open(filename, 'r') as file:
        data = json.load(file)

    if taco_data is None:
        taco_data = {
            "solutions": None,
            "input_output": None,
            "difficulty": None,
            "tags": None,
            "skill_types": None,
            "raw_tags": None,
            "date": None,
            "Expected Auxiliary Space": None,
            "Expected Time Complexity": None
        }
    if taco_data["solutions"]:
        data["solutions"] = taco_data["solutions"]
    if taco_data["input_output"]:
        data["input_output"] = taco_data["input_output"]
    data["difficulty"] = taco_data["difficulty"]
    data["tags"] = taco_data["tags"]
    data["skill_types"] = taco_data["skill_types"]
    data["raw_tags"] = taco_data["raw_tags"]
    data["date"] = taco_data["date"]
    data["Expected Auxiliary Space"] = taco_data["Expected Auxiliary Space"]
    data["Expected Time Complexity"] = taco_data["Expected Time Complexity"]

    with open(filename, 'w') as file:
        json.dump(data, file)


In [17]:
cf_matches = match(cf_dataset_dict)

  0%|          | 0/701 [00:00<?, ?it/s]

  0%|          | 0/1971 [00:00<?, ?it/s]

## Postprocess

In [36]:
def get_taco_data(taco_id):
    if taco_id.startswith("train_"):
        real_id = int(taco_id[6:])
        return taco_dataset["train"][real_id]
    elif taco_id.startswith("test_"):
        real_id = int(taco_id[5:])
        return taco_dataset["test"][real_id]
    else:
        raise ValueError()


In [23]:
# Copy files to new dataset
import shutil
import os
import json

destination_folder="/home/kaixin/Desktop/mmcode/mmcode_dataset"
for item in deduped_filtered_data:
    taco_id = item["taco_id"]
    crawled_path = item["crawled_path"]

    if "codeforces" in crawled_path or "aizu" in crawled_path:
        continue

    # Use the last folder name of the crawled_path as the new folder name
    new_folder_name = os.path.basename(os.path.normpath(crawled_path))
    new_folder_path = os.path.join(destination_folder, new_folder_name)

    # Create the new folder
    os.makedirs(new_folder_path, exist_ok=True)

    # Copy files from crawled_path to the new folder
    shutil.copytree(crawled_path, new_folder_path, dirs_exist_ok=True)

    # write taco data to a file
    taco_data = get_taco_data(taco_id)
    taco_json_path = os.path.join(new_folder_path, "taco.json")
    with open(taco_json_path, 'w') as f:
        json.dump(taco_data, f, indent=4)

    